# Demo — Auditing a Regulated Number to Source

A CSRD disclosure is only as trustworthy as its traceability to source and the consistency of its
cross-domain metric rules (**unit of measure**: kilograms vs. grams on `shipment.carbon_kg`;
**quarter attribution timestamp**: `orders.order_ts` vs. `shipment.ship_ts`). This walkthrough checks
upstream lineage with `gc.lineage.trace_upstream("quarterly_sustainability")`, runs the reporting
pipeline, compares the disclosed quarterly figures against recomputed figures in
`rep.reconciliation`, and demonstrates how to drill down from `rep.quarterly` into the underlying
order and shipment records in `rep.per_order`. The exercise asks you to trace a quarter to source,
isolate the anomalous rows driving the discrepancy, quantify the single-quarter and five-quarter
inflation, and write the traceability assessment.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog and S3 tables are seeded in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")
print("Upstream lineage of quarterly_sustainability:", gc.lineage.trace_upstream("quarterly_sustainability"))

rep = gc.run_sustainability_report()
display(rep.quarterly)
rep.reconciliation

## Drill down from a quarterly figure to `rep.per_order`
Notice in `rep.reconciliation` that **every quarter fails reconciliation** (`matches == False`):
`disclosed_carbon_kg` (which sums `shipment.carbon_kg` as-is) is 8x–65x larger than
`recomputed_carbon_kg` (`247,207.3 kg` disclosed vs. `5,483.0 kg` recomputed across all five quarters).

Use `rep.per_order` (`order_carbon`) to inspect the order-and-shipment grain (`order_ts` vs.
`ship_ts`, `carbon_kg`, and `quarter`) and compare the distribution of `carbon_kg` across shipments:

In [ ]:
print("Total non-cancelled shipped order rows:", len(rep.per_order))
print("\nDistribution of per-shipment carbon_kg across all quarters:")
print(rep.per_order["carbon_kg"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).round(2))
rep.per_order.head()

**Read the distribution:** up through the 95th percentile (`9.41 kg`), per-shipment emissions
sit in a normal single-digit kilogram range (`0.4`–`9.5 kg`), yet the 99th percentile and `max`
jump into the **thousands** (`9484.0 kg`), inflating `disclosed_carbon_kg` every quarter.

**Takeaway / your turn:** in the exercise you'll verify the upstream lineage of
`quarterly_sustainability`, filter `rep.per_order` on a specific quarter, isolate the exact rows
driving `delta_kg`, quantify both the single-quarter and five-quarter inflation, and write the
traceability assessment.